# AG08 · LangGraph-Style State Graphs and Durable Execution

**Track:** Agents (lesson 8 of 15) · **Time:** ~2.5 h · **Mode:** runs OFFLINE for free, or LIVE with `ANTHROPIC_API_KEY`
**Prereqs:** AG02 (agent loop), AG07 (workflow patterns) · **You will build:** a ~70-line graph engine with state, reducers, conditional edges, JSON checkpoints, time-travel and resume-after-crash, then port the AG07 content pipeline onto it

---
## 🎯 Why this matters
In AG07 you wrote workflows as ordinary Python functions: `draft = chain(...)`, `while not ok: revise(...)`. That is perfect until
the process **dies at step 7 of 9**. A Python call stack cannot be saved, so everything restarts: you re-pay for every LLM call and
you may repeat side effects (a second blog post, a second refund).

A **state graph** fixes this by making the control flow *data*: explicit state, named nodes, explicit edges. Once the workflow is
data, the runtime can write a **checkpoint after every node**, pause for a human, resume after a crash, and let you rewind and fork.
That is what *durable execution* means, and it is the real reason frameworks like LangGraph exist.

**Real scenario.** *Kiwi Trail Co.*, an outdoor retailer, generates a product-launch blog post for the new "Tarawera 2" rain jacket:
plan an outline, draft three sections, have an editor-model review it (fact and claim check), revise until it passes, then publish
to the CMS. The CMS has an outage during the publish step. After your fix, the pipeline must **resume at publish**, not start over,
and must **never publish twice**.

## 🧭 Learning objectives
By the end you can:
1. Model a workflow as **state + reducers + nodes + (conditional) edges** and explain what each buys you.
2. Implement a graph engine with atomic JSON checkpoints, **resume after a crash**, and time-travel (inspect and fork from any step).
3. Port the AG07 evaluator-optimizer pipeline onto the graph and **measure** the saved work after a crash (LLM calls re-run: 0).
4. Explain why durable execution forces **idempotent side effects**, and demonstrate the duplicate-publish bug and its fix.
5. Map every concept in your engine to the LangGraph API (guarded reference code), and say when you would *not* use a graph.

In [ ]:
# Setup: runs LIVE with ANTHROPIC_API_KEY, otherwise OFFLINE (scripted replay, free)
import sys, pathlib, subprocess
_find = lambda: next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "agentkit").is_dir()), None)
_root = _find()
if _root is None:                        # Colab / standalone download: fetch the repo helper package
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/nexageapps/AI.git", "_ai_repo"], check=False)
    _root = pathlib.Path("_ai_repo").resolve()
sys.path.insert(0, str(_root))
from agentkit import setup
setup()

## 1 · Concept: workflows as data

| Idea | What it is | Why it matters |
|---|---|---|
| **State** | one JSON-serialisable dict that flows through the whole run | the *only* thing that must be saved to resume |
| **Node** | `fn(state) -> partial update` (an LLM call, a tool call, plain code) | small, testable, replaceable units |
| **Reducer** | per-key merge rule: overwrite (default), append, add | lets nodes return *deltas*; makes parallel branches mergeable |
| **Edge** | static `A -> B` | the happy path |
| **Conditional edge** | `router(state) -> next node name` | loops (evaluator-optimizer) and branches (routing) |
| **Checkpoint** | state + "which node is next", saved after every node | resume, pause for humans, time-travel |

```
  __start__ ─▶ plan ─▶ write ─▶ review ──pass──▶ publish ─▶ __end__
                         ▲        │
                         └─ revise ◀── fail (and revisions < 2)
```

**Durable execution contract.** A node runs, *then* its result is checkpointed. If the process dies between the two, the node runs
**again** on resume. So nodes are executed **at least once**, and any node with a side effect must be **idempotent** (safe to repeat).
Keep this sentence in mind; Section 4 shows what happens when you ignore it.

**Question before we build:** which is cheaper to checkpoint, the whole message history or a small state dict? (Hint: what did the
nodes actually need to hand to each other?) Design the state *deliberately*; do not dump everything in it.

## 2 · Build it: a mini graph engine (≈70 lines)
Everything LangGraph does that matters for this lesson fits on a page. Read the code once, top to bottom; then we test it on a toy graph
before trusting it with LLM calls. Three design decisions to notice:
- `_save` serialises **first**, writes to a temp file, then `os.replace`s it: a crash mid-write can never leave a half-written checkpoint.
- Exceptions from a node propagate **without** advancing the checkpoint, so the last good state is always on disk.
- `interrupt_before` raises `Interrupt` *before* running a node: the hook for human approval (AG09 builds on this).

In [ ]:
import json, os, pathlib, tempfile, copy

END, START = "__end__", "__start__"

class Interrupt(Exception):
    """Raised before an `interrupt_before` node runs; resume with `invoke(None, thread, approve=True)`."""

class Graph:
    def __init__(self, reducers=None):
        self.reducers, self.nodes, self.edges, self.cond, self.entry = reducers or {}, {}, {}, {}, None
    def add_node(self, name, fn): self.nodes[name] = fn; return self
    def add_edge(self, a, b):
        if a == START: self.entry = b
        else: self.edges[a] = b
        return self
    def add_conditional_edges(self, a, router): self.cond[a] = router; return self
    def compile(self, root, interrupt_before=(), max_steps=25): return App(self, pathlib.Path(root), set(interrupt_before), max_steps)

class App:
    def __init__(self, g, root, interrupt_before, max_steps):
        self.g, self.root, self.interrupt_before, self.max_steps = g, root, interrupt_before, max_steps

    def _merge(self, state, update):                      # apply a node's partial update through the reducers
        new = copy.deepcopy(state)
        for k, v in (update or {}).items():
            new[k] = self.g.reducers[k](state.get(k), v) if k in self.g.reducers else v
        return new

    def _save(self, thread, step, node, nxt, state):      # atomic: serialise, write temp, rename
        blob = json.dumps({"step": step, "node": node, "next": nxt, "state": state})   # TypeError here => nothing written
        d = self.root / thread; d.mkdir(parents=True, exist_ok=True)
        tmp = d / f".tmp_{step:04d}"; tmp.write_text(blob); os.replace(tmp, d / f"step_{step:04d}.json")

    def history(self, thread):
        return [json.loads(p.read_text()) for p in sorted((self.root / thread).glob("step_*.json"))] if (self.root / thread).exists() else []

    def stream(self, inputs, thread, approve=False):
        """Run (or resume) a thread, yielding (node, update) after each node. inputs=None means 'resume from the last checkpoint'."""
        if inputs is not None:                             # fresh run
            if self.history(thread): raise ValueError(f"thread {thread!r} already has checkpoints; resume with inputs=None or use a new thread id")
            state = self._merge({}, inputs); step, nxt = 0, self.g.entry
            self._save(thread, step, START, nxt, state)
        else:                                              # resume
            cp = (self.history(thread) or [None])[-1]
            if cp is None: raise ValueError(f"no checkpoint for thread {thread!r}")
            state, step, nxt = cp["state"], cp["step"], cp["next"]
        for _ in range(self.max_steps):
            if nxt == END: return
            if nxt in self.interrupt_before and not approve: raise Interrupt(nxt)
            approve = False                                # approval covers exactly one node
            update = self.g.nodes[nxt](copy.deepcopy(state))
            state = self._merge(state, update)
            node, nxt = nxt, (self.g.cond[nxt](state) if nxt in self.g.cond else self.g.edges.get(nxt, END))
            step += 1; self._save(thread, step, node, nxt, state)
            yield node, update
        if nxt != END: raise RuntimeError(f"max_steps={self.max_steps} exceeded (loop in the graph?)")

    def invoke(self, inputs, thread, approve=False):
        for _ in self.stream(inputs, thread, approve): pass
        return self.state(thread)

    def state(self, thread): return self.history(thread)[-1]["state"]

    def fork(self, thread, step, new_thread, update=None):
        """Time-travel: start `new_thread` from checkpoint `step` of `thread`, optionally editing the state first."""
        cp = next(c for c in self.history(thread) if c["step"] == step)
        self._save(new_thread, cp["step"], cp["node"], cp["next"], self._merge(cp["state"], update))

### 2b · Test the engine on a toy graph *before* adding LLMs
A graph engine is infrastructure; you want its bugs found by cheap deterministic tests, not by an expensive agent run.
The toy graph is a retry-until-big-enough loop: `bump` adds 3 to a counter and logs it, and a conditional edge loops until `n >= 10`.
We check the reducer (`log` appends), the conditional edge, the checkpoint count, and the loop guard.

In [ ]:
TMP = pathlib.Path(tempfile.mkdtemp(prefix="ag08_"))          # all checkpoints live in a scratch dir
append = lambda old, new: (old or []) + new

toy = (Graph(reducers={"log": append})
       .add_node("bump", lambda s: {"n": s["n"] + 3, "log": [f"n={s['n'] + 3}"]})
       .add_edge(START, "bump")
       .add_conditional_edges("bump", lambda s: END if s["n"] >= 10 else "bump"))
toy_app = toy.compile(TMP / "toy")

final = toy_app.invoke({"n": 0}, thread="t1")
print(final)
assert final["n"] == 12 and final["log"] == ["n=3", "n=6", "n=9", "n=12"]     # reducer appended; router looped 4 times
assert len(toy_app.history("t1")) == 5                                          # step 0 (inputs) + one checkpoint per node run

# the loop guard: a router that never ends must not hang the process
spin = Graph().add_node("a", lambda s: {}).add_edge(START, "a").add_conditional_edges("a", lambda s: "a").compile(TMP / "spin", max_steps=7)
try:
    spin.invoke({}, "t")
    raise AssertionError("loop guard failed")
except RuntimeError as e:
    print("guard fired:", e)

## 3 · Use it: port the AG07 content pipeline
The pipeline from AG07's evaluator-optimizer pattern: `plan → write → review → (revise → review)* → publish`.
Compare with the loop version you wrote there: the logic is identical, but now each arrow is an edge, each box is a checkpointed node,
and `revisions` is explicit state instead of a local variable.

**State design** (all JSON-friendly): `brief`, `outline` (list of section titles), `sections` (title → text),
`feedback`, `revisions` (int, added by a reducer), `log` (appended), `post_url`.

The "LLM" is a `PolicyLLM` that plays three roles chosen by a `ROLE:` prefix in the system prompt. Offline it behaves like a believable
model: the first draft contains an *over-claim* ("guaranteed waterproof forever") that the editor rejects; the revision removes it.
With a key, the same nodes call real Claude. We count every LLM call, because **LLM calls are the cost we want durability to save**.

In [ ]:
from collections import Counter
from agentkit import get_llm, say

CALLS = Counter()
OUTLINE = ["Why we built it", "What changed vs the Tarawera 1", "Care and warranty"]

def policy(messages, system, tools):
    role, user = system.split(":")[0], messages[-1]["content"]
    if role == "PLANNER":
        return say(json.dumps(OUTLINE))
    if role == "WRITER":
        title = user.split("SECTION:")[1].split("\n")[0].strip()
        claim = "Our seam-taped shell is rated 20k mm and is guaranteed waterproof forever." if "FEEDBACK:" not in user else \
                "Our seam-taped shell is rated 20k mm and covered by a 2-year warranty against manufacturing faults."
        return say(f"{title}. The Tarawera 2 weighs 310 g and packs into its own pocket. {claim}")
    if role == "REVIEWER":
        bad = "guaranteed" in user.lower()
        return say(json.dumps({"pass": not bad, "feedback": "Remove absolute claims such as 'guaranteed'; cite the warranty terms instead." if bad else ""}))
    raise ValueError(role)

llm = get_llm(policy=policy)

def ask(role, prompt):
    CALLS[role] += 1
    return llm.chat([{"role": "user", "content": prompt}], system=f"{role}: you are a careful retail copywriter/editor. Reply concisely.").text

### 3b · The nodes
Each node is a *pure-ish* function of state returning a **delta**. Two things to notice:
- `review` combines an LLM judge with a **code-level** check (word budget). Invariants in code, judgment in the model (AG02).
- `publish` talks to a flaky CMS and uses an **idempotency key** derived from the content. We will break this on purpose later.

In [ ]:
class CMSOutage(Exception): pass
CMS = {"online": False, "posts": {}}                          # fake CMS; posts keyed by idempotency key

def cms_publish(slug, body, idem_key):
    if not CMS["online"]: raise CMSOutage("503 Service Unavailable from cms.kiwitrail.example")
    CMS["posts"].setdefault(idem_key, {"slug": slug, "body": body})          # same key twice => still one post
    return f"https://kiwitrail.example/blog/{slug}"

def plan(s):    return {"outline": json.loads(ask("PLANNER", f"Outline a launch post for: {s['brief']}")), "log": ["plan"]}
def write(s):
    fb = f"\nFEEDBACK: {s['feedback']}" if s.get("feedback") else ""
    return {"sections": {t: ask("WRITER", f"BRIEF: {s['brief']}\nSECTION: {t}{fb}") for t in s["outline"]}, "log": ["write"]}
def review(s):
    body = "\n".join(s["sections"].values())
    verdict = json.loads(ask("REVIEWER", body))
    if len(body.split()) > 220: verdict = {"pass": False, "feedback": "Too long: cut to 220 words."}      # invariant in code
    return {"feedback": verdict["feedback"], "ok": verdict["pass"], "revisions": 0 if verdict["pass"] else 1, "log": [f"review pass={verdict['pass']}"]}
def publish(s):
    import hashlib
    body = "\n\n".join(f"## {t}\n{x}" for t, x in s["sections"].items())
    key = hashlib.sha256(body.encode()).hexdigest()[:16]               # content-derived idempotency key
    return {"post_url": cms_publish("tarawera-2-launch", body, key), "log": ["publish"]}

route_after_review = lambda s: "publish" if s["ok"] else ("write" if s["revisions"] < 2 else END)   # END = give up, hand to a human

g = (Graph(reducers={"log": append, "revisions": lambda old, new: (old or 0) + new})
     .add_node("plan", plan).add_node("write", write).add_node("review", review).add_node("publish", publish)
     .add_edge(START, "plan").add_edge("plan", "write").add_edge("write", "review")
     .add_conditional_edges("review", route_after_review))

def mermaid(g):                                                     # graphs are data, so we can draw them
    lines = ["flowchart LR", f"  start([start]) --> {g.entry}"]
    lines += [f"  {a} --> {b}" for a, b in g.edges.items()]
    lines += [f"  {a} -.conditional.-> ?" for a in g.cond]
    return "\n".join(lines)
print(mermaid(g))

### 3c · Run 1: the CMS is down
Thread `launch-001` runs plan → write → review (fail) → write → review (pass) → **publish crashes**. Watch what is on disk afterwards:
the checkpoint after the last *successful* node, with `next == "publish"`.

In [ ]:
app = g.compile(TMP / "pipeline")
BRIEF = "Launch post for the Tarawera 2 rain jacket (310 g, 20k mm, recycled shell). Audience: day-walkers and trampers."

try:
    for node, upd in app.stream({"brief": BRIEF}, thread="launch-001"):      # stream() = see each node as it finishes
        print(f"  ✔ {node:<8} -> keys {sorted(upd)}")
except CMSOutage as e:
    print("  ✘ publish crashed:", e)

hist = app.history("launch-001")
print("\ncheckpoints on disk:", [(c["step"], c["node"], "→", c["next"]) for c in hist])
calls_before = sum(CALLS.values())
print("LLM calls so far:", dict(CALLS), "| total", calls_before)
assert hist[-1]["next"] == "publish" and not CMS["posts"]                  # stopped exactly before the side effect
assert calls_before == 9                                                    # plan 1 + write 3 + review 1 + rewrite 3 + review 1
assert app.state("launch-001")["revisions"] == 1

### 3d · Run 2: fix the outage, **resume** (no `inputs`)
`invoke(None, thread)` loads the last checkpoint and continues from `next`. The proof that this is real durable execution is a
*measurement*: the number of LLM calls made by the resume must be **zero** (only `publish` runs).

In [ ]:
CMS["online"] = True
final = app.invoke(None, thread="launch-001")
print(final["post_url"], "| log:", final["log"])
print("LLM calls made by the resume:", sum(CALLS.values()) - calls_before)
assert sum(CALLS.values()) == calls_before                    # zero LLM calls re-run: 8 calls' worth of work was not repeated
assert len(CMS["posts"]) == 1 and final["post_url"].endswith("tarawera-2-launch")
assert final["log"] == ["plan", "write", "review pass=False", "write", "review pass=True", "publish"]

### 3e · Time-travel and idempotency
Because every step is stored, you can *inspect* any past state and *fork* from it. Two uses:
**debugging** ("what did the draft look like when the editor rejected it?") and **what-if** ("rerun from step 3 with a stricter brief").
Here we also re-run `publish` from its checkpoint: a stand-in for "the process died right after the CMS call but before the checkpoint".
Because we used an idempotency key, the CMS still holds exactly one post.

In [ ]:
first_review = next(c for c in app.history("launch-001") if c["node"] == "review")
print("editor feedback at step", first_review["step"], "->", first_review["state"]["feedback"])
print("first draft excerpt ->", list(first_review["state"]["sections"].values())[0][:95], "...")
assert "guaranteed" in first_review["state"]["sections"]["Care and warranty"] and first_review["state"]["ok"] is False

pre_publish = next(c for c in app.history("launch-001") if c["next"] == "publish" and c["node"] == "review")
app.fork("launch-001", pre_publish["step"], "launch-001-replay")
app.invoke(None, thread="launch-001-replay")                  # runs publish a second time
print("posts in CMS after replaying publish:", len(CMS["posts"]))
assert len(CMS["posts"]) == 1                                  # idempotent: replay is harmless

# what-if: fork from the plan step with an edited brief; the LLM nodes run again because the state upstream changed
n0 = sum(CALLS.values())
app.fork("launch-001", 1, "launch-001-whatif", update={"brief": BRIEF + " Tone: playful."})
whatif = app.invoke(None, thread="launch-001-whatif")
print("what-if thread made", sum(CALLS.values()) - n0, "new LLM calls; brief now ends with:", whatif["brief"][-14:])
assert whatif["brief"].endswith("playful.") and sum(CALLS.values()) > n0

### 3f · Interrupt before a risky node (human-in-the-loop preview)
For a customer-facing publish, a human should click "approve". `interrupt_before=["publish"]` makes the graph pause *with a durable
checkpoint*: the approver can answer in five minutes or five days, from another process. Resume with `approve=True`.
(AG09 turns this into a full permissions and approval system.)

In [ ]:
CMS["posts"].clear()
gated = g.compile(TMP / "gated", interrupt_before=["publish"])
try:
    gated.invoke({"brief": BRIEF}, thread="launch-002")
except Interrupt as e:
    print(f"paused before {e.args[0]!r}; nothing published yet:", len(CMS["posts"]) == 0)
assert not CMS["posts"] and gated.history("launch-002")[-1]["next"] == "publish"

pending = gated.state("launch-002")                          # the reviewer inspects exactly what would be published
assert pending["ok"] is True and "guaranteed" not in " ".join(pending["sections"].values()).lower()
print("approver reads:", pending["sections"]["Care and warranty"])
done = gated.invoke(None, thread="launch-002", approve=True)  # a different process could do this later
print("published after approval:", done["post_url"])
assert len(CMS["posts"]) == 1

### 3g · Subgraphs: a graph as a node
A node is just `fn(state) -> update`, so a whole compiled graph can be a node. Useful for reusable sub-workflows (here: a two-step
*lint* that runs inside `review`-like stages in other pipelines). The outer state only sees the keys the wrapper chooses to map across,
which also keeps big inner state out of the outer checkpoints.

In [ ]:
lint = (Graph(reducers={"issues": append})
        .add_node("banned", lambda s: {"issues": [w for w in ("guaranteed", "best ever") if w in s["text"].lower()]})
        .add_node("length", lambda s: {"issues": ["too long"] if len(s["text"].split()) > 40 else []})
        .add_edge(START, "banned").add_edge("banned", "length")
        .compile(TMP / "lint"))
_n = iter(range(10**6))
def lint_node(s):                                              # wrapper maps outer state -> inner input -> outer update
    return {"lint_issues": lint.invoke({"text": s["text"]}, thread=f"l{next(_n)}")["issues"]}

outer = Graph().add_node("lint", lint_node).add_edge(START, "lint").compile(TMP / "outer")
print(outer.invoke({"text": "Guaranteed to keep you dry. Best ever."}, "o1")["lint_issues"])
assert outer.state("o1")["lint_issues"] == ["guaranteed", "best ever"]

### 3h · The same graph in LangGraph (reference: **check current LangGraph docs**)
Every piece of your engine has a direct counterpart. The mapping below is the point of this lesson; the exact import paths and
checkpointer class names do evolve, so treat code as a sketch and verify against the current documentation.

| Your engine | LangGraph |
|---|---|
| `Graph(reducers=...)` | a `TypedDict` state schema with `Annotated[list, operator.add]` reducers, passed to `StateGraph` |
| `add_node / add_edge / add_conditional_edges` | same names |
| `START`, `END` | `START`, `END` |
| `compile(root)` (JSON files) | `compile(checkpointer=...)` (in-memory, SQLite or Postgres savers) |
| `thread="launch-001"` | `config={"configurable": {"thread_id": "launch-001"}}` |
| `invoke(None, thread)` to resume | `invoke(None, config)` on a thread that has a checkpoint |
| `history`, `fork` | `get_state_history(config)` and `update_state` / re-invoking from a past checkpoint |
| `interrupt_before=[...]` | `compile(interrupt_before=[...])` or the newer `interrupt(...)` + `Command(resume=...)` pattern |
| `stream` | `stream(...)` with selectable stream modes |

The cell below is guarded: it only runs if `langgraph` is installed, and it uses trivial deterministic nodes so it needs no key.

In [ ]:
try:
    import operator
    from typing import Annotated, TypedDict
    from langgraph.graph import StateGraph, START as LG_START, END as LG_END
    from langgraph.checkpoint.memory import InMemorySaver
except ImportError:
    print("skipped: `pip install langgraph` to run this reference cell (the lesson's core does not need it)")
else:
    class PState(TypedDict):
        n: int
        log: Annotated[list, operator.add]          # reducer: append

    def bump(s: PState):
        return {"n": s["n"] + 3, "log": [f"n={s['n'] + 3}"]}

    b = StateGraph(PState)
    b.add_node("bump", bump)
    b.add_edge(LG_START, "bump")
    b.add_conditional_edges("bump", lambda s: LG_END if s["n"] >= 10 else "bump")
    lg_app = b.compile(checkpointer=InMemorySaver())
    cfg = {"configurable": {"thread_id": "t1"}}
    out = lg_app.invoke({"n": 0, "log": []}, cfg)
    print("LangGraph result:", out)
    assert out["n"] == 12 and out["log"] == ["n=3", "n=6", "n=9", "n=12"]     # identical behaviour to our toy engine
    print("checkpoints recorded:", len(list(lg_app.get_state_history(cfg))))

## 4 · Break it: failure modes of durable execution
Durability moves bugs, it does not remove them. Three classic ones, each with a failing demo and a fix.

**4a · State that cannot be serialised.** A `datetime`, a `set`, a client object, or a numpy array in state breaks checkpointing.
Because `_save` serialises *before* writing, the previous checkpoint survives; verify that.

In [ ]:
import datetime
bad = (Graph().add_node("stamp", lambda s: {"when": datetime.datetime(2025, 1, 1)}).add_node("next", lambda s: {})
       .add_edge(START, "stamp").add_edge("stamp", "next").compile(TMP / "bad"))
try:
    bad.invoke({"x": 1}, "t")
    raise AssertionError("expected a serialisation failure")
except TypeError as e:
    print("TypeError:", e)
assert [c["step"] for c in bad.history("t")] == [0]            # only the input checkpoint exists: no corrupt/partial file
# fix: store primitives (ISO strings), re-hydrate inside nodes
ok = Graph().add_node("stamp", lambda s: {"when": datetime.datetime(2025, 1, 1).isoformat()}).add_edge(START, "stamp").compile(TMP / "good")
assert ok.invoke({"x": 1}, "t")["when"] == "2025-01-01T00:00:00"

**4b · The duplicate side effect.** The nastiest crash happens *after* the CMS accepted the post but *before* the checkpoint was written.
On resume the node runs again. With a naive `publish` (no idempotency key) you now have two identical posts. We simulate the crash
with `SimulatedKill`, a `BaseException`, standing in for `kill -9` / OOM / a pre-empted container.

In [ ]:
class SimulatedKill(BaseException): pass

NAIVE = []                                                      # naive CMS: appends every call
kill_once = {"armed": True}
def naive_publish(s):
    NAIVE.append(s["title"])                                    # side effect happens...
    if kill_once["armed"]:
        kill_once["armed"] = False; raise SimulatedKill()       # ...then the process dies before the checkpoint
    return {"done": True}

naive_app = Graph().add_node("publish", naive_publish).add_edge(START, "publish").compile(TMP / "naive")
try: naive_app.invoke({"title": "Tarawera 2 launch"}, "t")
except SimulatedKill: print("process killed after the side effect")
naive_app.invoke(None, "t")                                     # resume => publish runs again
print("posts:", NAIVE)
assert NAIVE == ["Tarawera 2 launch"] * 2                       # DUPLICATE: the bug we predicted

# fix: make the effect idempotent (key = hash of what the effect *means*), so at-least-once becomes effectively-once
SAFE, kill_once["armed"] = {}, True
def safe_publish(s):
    SAFE.setdefault(("post", s["title"]), "live")               # keyed write: repeating it changes nothing
    if kill_once["armed"]:
        kill_once["armed"] = False; raise SimulatedKill()
    return {"done": True}
safe_app = Graph().add_node("publish", safe_publish).add_edge(START, "publish").compile(TMP / "safe")
try: safe_app.invoke({"title": "Tarawera 2 launch"}, "t")
except SimulatedKill: pass
safe_app.invoke(None, "t")
assert len(SAFE) == 1 and safe_app.state("t")["done"]
print("with an idempotency key: posts =", len(SAFE))

**4c · Other traps to know (discuss, then test one yourself in the exercises).**
- **Infinite loops**: a conditional edge that never returns `END`. We saw `max_steps`; LangGraph has a similar recursion limit. Put a *business* counter (`revisions < 2`) in state too, so the graph ends in a defined, human-handoff state.
- **State drift**: you deploy new code that renames a state key or removes a node while old threads are paused. Version your state schema and migrate (or fail loudly) on resume.
- **Reducer surprises**: a key without a reducer is *overwritten*; two parallel branches writing it silently lose one write. Give shared keys a merge rule.
- **Over-large state**: every checkpoint copies state. Store big blobs (documents, images) elsewhere and keep ids in state.

**When NOT to use a graph.** A linear 3-step chain with no side effects, no pause and no loop is clearer as a function. Reach for a graph
when you need *any* of: loops with a durable counter, long waits for humans, crash recovery, replay/debugging, or branching you want to
draw and review.

## 5 · Exercises
**Practice (30 min)**
1. Add a node `seo_check` between `write` and `review` that fails the run if the title section lacks the word "Tarawera". Wire it with a conditional edge back to `write`. Assert the graph still publishes exactly one post.
2. Use `app.history(...)` to print a table of `(step, node, next, len(log))` for `launch-001`. Which step number is the *first* review?

**Stretch (60 min)**
3. Add a `retry(fn, attempts, on=(CMSOutage,))` wrapper for nodes. Make the CMS fail twice then succeed; assert `publish` succeeded in a single `invoke` and that retries did **not** add checkpoints. Why should a retry wrapper *not* catch every exception?
4. Add `Graph.fan_out(a, [b, c], join=d)` semantics (run `b` and `c` on the same input, merge their updates with the reducers, then `d`). Which key needs a reducer for this to be safe?

**Challenge (2 h)**
5. Add *schema versioning*: write `"schema": 1` into every checkpoint; on resume with a different engine version, call a user-supplied `migrate(state, from_v, to_v)`. Test with a checkpoint written under v1 that renames `feedback` to `editor_notes`.

In [ ]:
# Starter for exercise 1 - uncomment and complete; the assertions are the self-check.
def seo_check(s):
    # TODO: return {"ok": False, "feedback": "Title section must mention Tarawera", "revisions": 1} if missing, else {}
    raise NotImplementedError

# g2 = (Graph(reducers=g.reducers).add_node("plan", plan).add_node("write", write).add_node("seo", seo_check)
#       .add_node("review", review).add_node("publish", publish)
#       .add_edge(START, "plan").add_edge("plan", "write").add_edge("write", "seo")
#       .add_conditional_edges("seo", lambda s: "write" if s.get("ok") is False else "review")
#       .add_conditional_edges("review", route_after_review))
# CMS["posts"].clear(); r = g2.compile(TMP / "g2").invoke({"brief": BRIEF}, "t")
# assert len(CMS["posts"]) == 1

## ✅ Checkpoint (answer before moving on)
1. Why must a node's checkpoint be written *after* the node returns, and what does that imply about side effects?
2. What is a reducer, and what silently goes wrong in a parallel fan-out if a shared key has none?
3. In Section 3d the resume made 0 LLM calls. Name two things that would have made it make more.
4. Your team wants an "approve before sending" step that may take days. Which engine feature gives you that, and what must be true of the stored state?

<details><summary>Answers</summary>

1. Because the checkpoint records *completed* work; if it were written before, a crash would skip the node forever. The consequence is at-least-once execution: a crash between the effect and the checkpoint repeats the node, so side effects must be idempotent (keys, upserts, dedup tables).
2. A reducer is a per-key merge function `(old, new) -> merged`. Without one the key is overwritten, so with two parallel writers one result is lost with no error.
3. (a) Resuming from `inputs` instead of `None` / using a new thread id, so nothing is loaded; (b) a changed upstream state (e.g. forking with an edited brief), or not checkpointing the LLM nodes because they were fused into one big node that crashed.
4. `interrupt_before` (or LangGraph's `interrupt`) with a durable checkpoint; the state must be fully serialisable and contain everything the approver needs, because the process that resumes may be a different one, days later.
</details>

## 📌 Key takeaways
- A workflow becomes durable when control flow is **data**: state + nodes + edges, checkpointed after every node.
- Durable execution is **at-least-once**; correctness comes from **idempotent side effects**, not from the framework.
- Checkpoints give you four things for the price of one: **resume, pause-for-human, time-travel, and replay-based debugging**.
- Measure durability: *LLM calls re-run after a crash* should be 0; *duplicate side effects* should be 0.
- Use a graph when you need loops, waits, recovery or reviewability; use a plain function otherwise.

## 📚 Further reading
- LangGraph documentation (concepts: state, persistence/checkpointers, human-in-the-loop, time travel): langchain-ai.github.io/langgraph
- Anthropic Engineering, *Building effective agents*: anthropic.com/engineering/building-effective-agents
- Malewicz et al., *Pregel: A System for Large-Scale Graph Processing* (SIGMOD 2010), the bulk-synchronous model LangGraph's runtime is described as being inspired by.

**Next → AG09 · Human-in-the-Loop, Permissions and Guardrails**: turn the `interrupt_before` hook into a real policy engine with allow/ask/deny rules, guardrails and an audit log.